# Prompt Versioning

Version prompts in the Opik Prompt Library, compare versions for hallucination with side-by-side experiments, and run traced inference against the winning prompt version via LiteLLM.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-ml/opik-examples/blob/main/guides/prompt_versioning/prompt_versioning.ipynb)

Every call to `client.create_prompt(name=..., prompt=...)` with the same `name` creates a new, immutable **commit** rather than overwriting existing prompts — giving you a full audit history and the ability to fetch any historical version by its commit hash.

We follow a single prompt — an earnings-call summarizer — through the whole loop: commit two versions, fetch each by hash, compare them for hallucination as side-by-side experiments, then run the winning version on a new transcript.

**What you'll learn:**

- How to commit prompt versions with descriptive `change_description` labels using `client.create_prompt()`
- How to fetch specific prompt commits (by hash) or retrieve the newest version (`client.get_prompt()`)
- How to run side-by-side prompt evaluation (`evaluate_prompt`) with an LLM-as-judge `Hallucination` metric to compare prompt versions as experiments in the Opik UI
- How to run inference using `litellm` traced with `@opik.track(project_name=...)`


In [ ]:
%pip install --quiet --upgrade opik litellm

In [ ]:
import os
import litellm
import opik
from opik.evaluation import evaluate_prompt
from opik.evaluation.metrics import Hallucination

OPIK_PROJECT_NAME = "prompt-versioning"

# Credentials come from OPIK_API_KEY / OPIK_WORKSPACE in the environment; targets Opik Cloud.
# install_mcp=False keeps configure non-interactive for headless environments.
opik.configure(project_name=OPIK_PROJECT_NAME, install_mcp=False)

# CI sets OPIK_EXAMPLES_MODEL to a cheap model.
MODEL_NAME = os.environ.get("OPIK_EXAMPLES_MODEL", "openai/gpt-5.6-sol")


## 1. Creating and committing prompt versions

Every call to `client.create_prompt()` with the same `name` creates an immutable version ("commit") of that prompt in the Opik Prompt Library. Nothing is overwritten — one name, many commits.

We pass `change_description=...` to document why each version was created — this renders directly in the Opik UI.


In [ ]:
SUMMARIZER_V1 = """You are a financial analyst summarizing earnings calls.
Provide a comprehensive summary including key metrics, guidance, and management commentary."""

SUMMARIZER_V2 = """You are a financial analyst creating earnings call summaries for compliance-reviewed reports.

## Strict Rules
- ONLY include facts explicitly stated in the provided transcript
- Use EXACT numbers - never round or approximate
- Never infer sentiment not directly expressed by management
- If guidance wasn't mentioned, state "No guidance provided"
- Attribute all quotes: "CEO [Name] stated..."

## Output Format
**Reported Metrics**: [Only numbers explicitly stated]
**Management Commentary**: [Direct quotes or close paraphrases only]
**Forward Guidance**: [Only if explicitly provided]
**NOT MENTIONED**: [List key items not covered]

If uncertain whether something was stated, DO NOT include it."""

client = opik.Opik()

PROMPT_NAME = "earnings-call-summarizer"

# Commit Version 1
v1_prompt = client.create_prompt(
    name=PROMPT_NAME,
    prompt=SUMMARIZER_V1,
    change_description="Baseline summarizer prompt with generic instructions",
)
print(f"Created '{PROMPT_NAME}' commit v1: {v1_prompt.commit}")

# Commit Version 2
v2_prompt = client.create_prompt(
    name=PROMPT_NAME,
    prompt=SUMMARIZER_V2,
    change_description="Strict summarizer prompt with factual and formatting rules",
)
print(f"Created '{PROMPT_NAME}' commit v2: {v2_prompt.commit}")


## 2. Fetching prompt versions (by hash and by latest)

`client.get_prompt(name=...)` retrieves prompt versions from the Prompt Library:
- Omit `commit` to get the **latest** committed version.
- Pass `commit="<hash>"` to fetch a **specific historical version**.


In [ ]:
# Fetch the latest committed version (v2)
latest_prompt = client.get_prompt(name=PROMPT_NAME)
print(f"Latest commit: {latest_prompt.commit}")
print(f"Change description: {latest_prompt.change_description}")

# Fetch the specific historical v1 commit by hash — still there, unchanged
historical_v1 = client.get_prompt(name=PROMPT_NAME, commit=v1_prompt.commit)
print(f"Fetched historical v1 commit: {historical_v1.commit}")
print(f"Historical v1 description: {historical_v1.change_description}")
assert historical_v1.prompt == SUMMARIZER_V1


## 3. Comparing prompt versions via side-by-side evaluation

Before shipping a new prompt version, evaluate both versions on an Opik dataset using `evaluate_prompt()` and an LLM-as-judge metric like `Hallucination` — summary vs. source transcript is exactly what this metric is built to catch.

Each call to `evaluate_prompt()` logs a named **experiment** in Opik linked to that specific prompt commit. Open the Opik UI to compare hallucination scores side-by-side and decide which version to ship.


In [ ]:
DATASET_NAME = "earnings-call-summarizer-eval"

TRANSCRIPT = (
    "Apple reported Q4 revenue of $89.5 billion, up 6% year-over-year. iPhone revenue grew "
    "10% to $43.8 billion. CEO Tim Cook said 'We\'re thrilled with the strong demand for "
    "iPhone 15 Pro.'"
)
CONTEXT = "Q4 revenue: $89.5B, +6% YoY. iPhone: $43.8B, +10%. Tim Cook commented on iPhone 15 Pro demand."
QUERY = f"Summarize this earnings call:\n{TRANSCRIPT}"

dataset = client.get_or_create_dataset(name=DATASET_NAME, project_name=OPIK_PROJECT_NAME)
dataset.insert([{"input": QUERY, "context": CONTEXT}])


def score_prompt_version(prompt: opik.Prompt, experiment_name: str):
    return evaluate_prompt(
        dataset=dataset,
        messages=[
            {"role": "system", "content": prompt.prompt},
            {"role": "user", "content": "{{input}}"},
        ],
        model=MODEL_NAME,
        scoring_metrics=[Hallucination(model=MODEL_NAME)],
        experiment_name=experiment_name,
        prompt=prompt,
        project_name=OPIK_PROJECT_NAME,
    )


exp1 = score_prompt_version(v1_prompt, "earnings-call-summarizer-v1-baseline")
exp2 = score_prompt_version(v2_prompt, "earnings-call-summarizer-v2-strict")

print("Evaluations completed. Compare experiment scores side-by-side in the Opik UI.")


## 4. Running traced inference with the winning version

The stricter, compliance-reviewed v2 scores lower on hallucination — that's the version to ship. Fetch the latest committed prompt dynamically from Opik and run it via LiteLLM on a brand-new transcript, traced with `@opik.track(project_name=...)`.

Because your application code fetches the latest commit by name rather than hardcoding a string template, promoting a new prompt version in Opik changes what your application runs without a code deployment.


In [ ]:
@opik.track(project_name=OPIK_PROJECT_NAME)
def run_inference(system_prompt: str, user_query: str) -> str:
    response = litellm.completion(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
    )
    return response.choices[0].message.content


NEW_TRANSCRIPT = (
    "Microsoft reported Q2 revenue of $62.0 billion, up 18% year-over-year. Azure and other "
    "cloud services revenue grew 30%. CFO Amy Hood said guidance for next quarter assumes "
    "continued double-digit cloud growth."
)
NEW_QUERY = f"Summarize this earnings call:\n{NEW_TRANSCRIPT}"

# Fetch the latest prompt version and run inference — no hardcoded prompt text here
current_prompt = client.get_prompt(name=PROMPT_NAME)
answer = run_inference(current_prompt.prompt, NEW_QUERY)

print(f"Using prompt '{PROMPT_NAME}' commit {current_prompt.commit[:8]}...\n")
print(answer)


## Summary

| Action | SDK Method | Key Feature |
|---|---|---|
| Create version | `client.create_prompt(name=..., prompt=..., change_description=...)` | Immutable commit hash created; `change_description` labels version purpose |
| Fetch latest | `client.get_prompt(name=...)` | Returns newest commit without hardcoding text in app code |
| Fetch commit | `client.get_prompt(name=..., commit="<hash>")` | Retrieves exact historical prompt text |
| Evaluate side-by-side | `evaluate_prompt(..., prompt=prompt, experiment_name=...)` | Creates side-by-side experiments comparing metrics across prompt commits |
| Trace inference | `@opik.track(project_name=...)` + `litellm.completion()` | Captures LLM calls in Opik project |

**Key takeaways:**

1. Prompts are immutable — calling `create_prompt` with an existing name creates a new commit hash. Don't bake the version into the name (e.g. avoid `earnings-summarizer-v1`); one name, many commits.
2. Use `change_description` to label why each version was created for clean rendering in the Opik UI.
3. Pass `prompt=prompt` to `evaluate_prompt()` so experiment metrics in Opik are directly linked to specific prompt commits — compare before you ship.
4. Decouple application code from prompt text by fetching the latest commit dynamically via `client.get_prompt()`, so the version you evaluated is the version you run.
